# Checkpoint 4F — Multi-Satellite Footprint Consistency (calibration-cautious)

**Question:** *do independently measured, threshold-defined high-flux footprints appear in broadly the
same location across the available POES/MetOp satellites for January 2024?* This is a
**footprint-location/shape consistency** study — **NOT** an absolute-intensity comparison and **NOT**
a claim that any satellite is more correct.

- Month: Jan 2024 · Satellites: noaa15, noaa18, noaa19, metop01, metop03
- Channel `mep_omni_flux_p1` (differential proton flux ~25 MeV, `#/cm2-s-str-MeV` if confirmed)
- Region lat[-70,20]xlon[-100,20]; lon [0,360)->[-180,180) · grids 5/2deg · stats mean/median ·
  thresholds top 20/10/5/2/1% · drop `mep_IFC_on==1`, keep `==-1` (uninterpreted)

**Caveat:** satellite-to-satellite differences may reflect calibration, instrument aging, detector
response, orbital/local-time sampling, or platform differences. Absolute flux is **not** cross-calibrated
(`absolute_flux_comparison_allowed=False` on every row). No SAA boundary / dose / health / danger /
discovery claims.

In [1]:
import sys, json, itertools
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from saa.satellite_analysis import (
    build_satellite_audit, satellite_family, run_multisatellite_sensitivity,
    pairwise_centroid_distances, save_table,
    plot_satellite_mean_map, plot_satellite_sample_count, plot_satellite_comparison,
    plot_multisatellite_centroids, plot_pairwise_distance_heatmap,
)
from saa.aggregate import load_range, build_grid_table, add_coverage_mask
from saa.grid_flux import prepare_region
from saa.load_poes import download_poes_file, open_poes_netcdf, channel_metadata, DEFAULT_CHANNEL
from saa.threshold_analysis import haversine_km

RAW = ROOT / "data" / "raw"; PROC = ROOT / "data" / "processed"
TBL = ROOT / "outputs" / "tables"; FIG = ROOT / "outputs" / "figures"
for d in (PROC, TBL, FIG): d.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220, "display.max_columns", 40)
SATELLITES = ["noaa15", "noaa18", "noaa19", "metop01", "metop03"]
COVERAGE_THRESHOLD = 30  # consistent across satellites (same as CP4A/CP4E); coverage reported below
print("ROOT:", ROOT, "| satellites:", SATELLITES)

ROOT: /Users/fellipegoncalvesleite/saa-poes-mapping | satellites: ['noaa15', 'noaa18', 'noaa19', 'metop01', 'metop03']


## 1. Reuse + extend the CP4E audit (re-confirm compatibility for all candidates)

In [2]:
# reuse the accepted CP4E audit table as the starting point
cp4e_audit = pd.read_parquet(TBL / "cp4e_satellite_availability_audit.parquet")
print("CP4E audit:")
print(cp4e_audit[["satellite","available_days","missing_days","opens_with_loader","has_mep_omni_flux_p1",
                  "has_mep_IFC_on","channel_units","recommended_for_pilot"]].to_string(index=False))

# re-confirm now (fresh archive listing + sample open) and extend with a CP4F verdict
audit = build_satellite_audit(SATELLITES, raw_dir=str(RAW))
audit["family_or_platform"] = audit["satellite"].map(satellite_family)
compatible, excluded = [], []
for _, r in audit.iterrows():
    ok = bool(r.opens_with_loader and r.has_time_lat_lon and r.has_mep_omni_flux_p1 and r.has_mep_IFC_on
              and r.available_days == r.expected_days
              and r.recommended_for_pilot in ("eligible", "reference"))
    (compatible if ok else excluded).append(r.satellite)
audit["cp4f_included"] = audit.satellite.isin(compatible)
save_table(audit, TBL / "cp4f_satellite_compatibility.csv", TBL / "cp4f_satellite_compatibility.parquet")
print("\nCP4F re-confirmation:")
print(audit[["satellite","family_or_platform","available_days","opens_with_loader","has_mep_omni_flux_p1",
             "channel_long_name","cp4f_included"]].to_string(index=False))
print("\nCOMPATIBLE (included):", compatible)
print("EXCLUDED:", excluded or "none")

CP4E audit:
satellite  available_days missing_days  opens_with_loader  has_mep_omni_flux_p1  has_mep_IFC_on   channel_units recommended_for_pilot
  metop01              31         none               True                  True            True #/cm2-s-str-MeV              eligible
  metop03              31         none               True                  True            True #/cm2-s-str-MeV              eligible
   noaa15              31         none               True                  True            True #/cm2-s-str-MeV              eligible
   noaa18              31         none               True                  True            True #/cm2-s-str-MeV              eligible
   noaa19              31         none               True                  True            True #/cm2-s-str-MeV             reference



CP4F re-confirmation:
satellite family_or_platform  available_days  opens_with_loader  has_mep_omni_flux_p1                                                channel_long_name  cp4f_included
  metop01              MetOp              31               True                  True MEPED proton differential flux at 25 MeV omnidirection telescope           True
  metop03              MetOp              31               True                  True MEPED proton differential flux at 25 MeV omnidirection telescope           True
   noaa15          NOAA-POES              31               True                  True MEPED proton differential flux at 25 MeV omnidirection telescope           True
   noaa18          NOAA-POES              31               True                  True MEPED proton differential flux at 25 MeV omnidirection telescope           True
   noaa19          NOAA-POES              31               True                  True MEPED proton differential flux at 25 MeV omnidirection telesc

## 2-3. Process monthly regional data + build 5deg/2deg grids for each compatible satellite

In [3]:
regional, grids, meta = {}, {}, {}
report = []
for sat in compatible:
    # ensure files present (cached; retry transient errors via timeout-hardened loader)
    import urllib.error
    for d in pd.date_range("2024-01-01", "2024-01-31", freq="D"):
        for attempt in range(4):
            try: download_poes_file(d.date(), sat, output_dir=str(RAW)); break
            except (urllib.error.URLError, urllib.error.HTTPError, OSError):
                if attempt == 3: raise
    rl = load_range("2024-01-01", "2024-01-31", satellite=sat, raw_dir=str(RAW))
    reg, counts = prepare_region(rl.df)
    reg.to_parquet(PROC / f"cp4f_{sat}_2024-01_mep_omni_flux_p1_region.parquet", index=False)
    regional[sat] = reg
    # channel metadata from a real file
    with open_poes_netcdf(rl.paths[0]) as ds:
        meta[sat] = channel_metadata(ds, [DEFAULT_CHANNEL])[DEFAULT_CHANNEL]
    g = {}
    for gd in (5.0, 2.0):
        mask_col = f"enough_samples_{int(gd)}deg"
        t = add_coverage_mask(build_grid_table(reg, lat_step=gd, lon_step=gd), COVERAGE_THRESHOLD, mask_col)
        t.to_parquet(TBL / f"cp4f_{sat}_2024-01_grid_{int(gd)}deg.parquet", index=False)
        g[int(gd)] = t
    grids[sat] = g
    report.append({"satellite": sat, "loaded_days": rl.n_loaded, "missing": rl.missing_dates or "none",
                   "region_rows": len(reg),
                   "cells5_pass": int(g[5][f"enough_samples_5deg"].sum()),
                   "sc5_med": float(g[5]["sample_count"].median()), "sc5_min": int(g[5]["sample_count"].min()),
                   "cells2_pass": int(g[2][f"enough_samples_2deg"].sum()),
                   "sc2_med": float(g[2]["sample_count"].median()), "sc2_min": int(g[2]["sample_count"].min()),
                   "p1_units": meta[sat]["units"]})
rep = pd.DataFrame(report)
print(rep.to_string(index=False))
print("\nchannel long_name per satellite:")
for s in compatible: print(f"  {s}: {meta[s]['long_name']!r}")

satellite  loaded_days missing  region_rows  cells5_pass  sc5_med  sc5_min  cells2_pass  sc2_med  sc2_min        p1_units
  metop01           31    none       227572          432    528.0      450         2700     85.0       55 #/cm2-s-str-MeV
  metop03           31    none       226645          432    526.0      434         2700     82.0       50 #/cm2-s-str-MeV
   noaa15           31    none       207232          432    485.0      293         2665     78.0        1 #/cm2-s-str-MeV
   noaa18           31    none       208722          432    487.0      236         2571     78.0        1 #/cm2-s-str-MeV
   noaa19           31    none       205153          432    477.0      276         2685     76.0       16 #/cm2-s-str-MeV

channel long_name per satellite:
  metop01: 'MEPED proton differential flux at 25 MeV omnidirection telescope'
  metop03: 'MEPED proton differential flux at 25 MeV omnidirection telescope'
  noaa15: 'MEPED proton differential flux at 25 MeV omnidirection telescope'
 

Coverage threshold `>=30` is applied consistently to all satellites (same as CP4A/CP4E). The
per-satellite coverage above is reported so any poorer-coverage satellite (e.g. the older NOAA-15) is
visible rather than hidden; low-sample cells are excluded by the mask and so cannot dominate the
centroid/area calculations.

## 4. Multi-satellite threshold sensitivity (compatible_count x 2 x 2 x 5 rows)

In [4]:
note = ("CP4F multi-satellite footprint-location consistency; same SEM-2/MEPED p1 metadata; "
        "absolute flux NOT cross-calibrated; mep_IFC_on==-1 uninterpreted")
specs = [{"satellite": sat, "note": note, "grids": [
            {"grid_deg": 5, "step": 5.0, "table": grids[sat][5], "mask_col": "enough_samples_5deg", "coverage_threshold": COVERAGE_THRESHOLD},
            {"grid_deg": 2, "step": 2.0, "table": grids[sat][2], "mask_col": "enough_samples_2deg", "coverage_threshold": COVERAGE_THRESHOLD}]}
         for sat in compatible]
sens = run_multisatellite_sensitivity(specs)
save_table(sens, TBL / "cp4f_multisatellite_threshold_sensitivity.csv",
           TBL / "cp4f_multisatellite_threshold_sensitivity.parquet")
print("rows:", len(sens), "(expected", len(compatible)*2*2*5, ") | satellites:", sorted(sens.satellite.unique()))
print("absolute_flux_comparison_allowed unique:", sens.absolute_flux_comparison_allowed.unique(),
      "| selected_cell_count min:", int(sens.selected_cell_count.min()))
print(sens[(sens.grid_deg==5)&(sens.statistic_used=='mean_flux')&(sens.threshold_label=='top10')]
      [["satellite","satellite_family_or_platform","cells_available_after_coverage_mask","selected_cell_count",
        "selected_area_km2","centroid_lat_flux_weighted","centroid_lon_flux_weighted","peak_flux"]]
      .rename(columns={"satellite_family_or_platform":"fam"}).to_string(index=False))

rows: 100 (expected 100 ) | satellites: ['metop01', 'metop03', 'noaa15', 'noaa18', 'noaa19']
absolute_flux_comparison_allowed unique: [False] | selected_cell_count min: 5
satellite       fam  cells_available_after_coverage_mask  selected_cell_count  selected_area_km2  centroid_lat_flux_weighted  centroid_lon_flux_weighted  peak_flux
  metop01     MetOp                                  432                   44       1.254069e+07                  -21.667350                  -54.950782  26.725041
  metop03     MetOp                                  432                   44       1.256828e+07                  -21.390826                  -55.589622  28.401419
   noaa15 NOAA-POES                                  432                   44       1.254767e+07                  -20.964353                  -52.888098  77.763193
   noaa18 NOAA-POES                                  432                   44       1.261403e+07                  -20.914793                  -55.604691  36.192966
   noaa19

## 5. Pairwise flux-weighted centroid distances (4 main cases)

In [5]:
cases = [("top10_5deg_mean", 5, "mean_flux", "top10"),
         ("top5_5deg_mean",  5, "mean_flux", "top5"),
         ("top10_2deg_mean", 2, "mean_flux", "top10"),
         ("top5_2deg_mean",  2, "mean_flux", "top5")]
pairwise = pairwise_centroid_distances(sens, cases)
save_table(pairwise, TBL / "cp4f_pairwise_centroid_distances.csv",
           TBL / "cp4f_pairwise_centroid_distances.parquet")
print("pairwise rows:", len(pairwise))
for case, *_ in cases:
    sub = pairwise[pairwise.comparison_case == case]
    print(f"\n== {case} == max {sub.distance_km.max():.0f} km | median {sub.distance_km.median():.0f} km")
    print(sub[["satellite_a","satellite_b","distance_km"]].sort_values("distance_km", ascending=False).head(4).to_string(index=False))

pairwise rows: 40

== top10_5deg_mean == max 284 km | median 105 km
satellite_a satellite_b  distance_km
    metop03      noaa15   284.090147
     noaa15      noaa18   282.172375
     noaa15      noaa19   269.242004
    metop01      noaa15   227.516445

== top5_5deg_mean == max 453 km | median 126 km
satellite_a satellite_b  distance_km
    metop01      noaa15   453.462455
    metop03      noaa15   435.855806
     noaa15      noaa18   427.566716
     noaa15      noaa19   407.023960

== top10_2deg_mean == max 265 km | median 81 km
satellite_a satellite_b  distance_km
    metop03      noaa15   264.599321
     noaa15      noaa19   245.408179
    metop01      noaa15   227.669888
     noaa15      noaa18   225.200333

== top5_2deg_mean == max 387 km | median 85 km
satellite_a satellite_b  distance_km
    metop03      noaa15   386.867576
    metop01      noaa15   353.508678
     noaa15      noaa18   320.136041
     noaa15      noaa19   316.479838


## 6. Consistency summary + comparison to CP4B threshold sensitivity magnitude

In [6]:
def maxspread(case):
    return pairwise[pairwise.comparison_case == case].distance_km.max()
print("== maximum pairwise centroid spread across satellites (flux-weighted) ==")
for case, *_ in cases:
    print(f"  {case}: {maxspread(case):.0f} km")

# NOAA family vs MetOp family (top10 5deg mean)
sub = sens[(sens.grid_deg==5)&(sens.statistic_used=='mean_flux')&(sens.threshold_label=='top10')]
for fam in ["NOAA-POES", "MetOp"]:
    f = sub[sub.satellite_family_or_platform == fam]
    if len(f):
        print(f"  {fam}: centroids " + ", ".join(f"{r.satellite}({r.centroid_lat_flux_weighted:.1f},{r.centroid_lon_flux_weighted:.1f})" for r in f.itertuples()))

# NOAA-15 closeness check (oldest satellite)
if "noaa15" in compatible:
    d15 = pairwise[(pairwise.comparison_case=="top10_5deg_mean") &
                   ((pairwise.satellite_a=="noaa15")|(pairwise.satellite_b=="noaa15"))]
    print(f"\n  NOAA-15 top10/5deg centroid distance to others: "
          f"min {d15.distance_km.min():.0f} km, max {d15.distance_km.max():.0f} km")

# compare to CP4B threshold sensitivity (intra-satellite centroid shift top20->top1)
cp4b = pd.read_parquet(TBL / "cp4b_threshold_sensitivity.parquet")
c = cp4b[(cp4b.grid_deg==5)&(cp4b.statistic_used=='mean_flux')].sort_values("percentile_cutoff")
t20 = c.iloc[0]; t1 = c.iloc[-1]
cp4b_shift = haversine_km(t20.centroid_lat_flux_weighted, t20.centroid_lon_flux_weighted,
                          t1.centroid_lat_flux_weighted, t1.centroid_lon_flux_weighted)
print(f"\n== magnitude check ==")
print(f"  inter-satellite spread (top10 5deg mean): {maxspread('top10_5deg_mean'):.0f} km")
print(f"  CP4B intra-satellite threshold shift top20->top1 (5deg mean): {cp4b_shift:.0f} km")
print("  -> inter-satellite footprint spread is SMALL relative to the methodological threshold effect"
      if maxspread('top10_5deg_mean') < cp4b_shift else "  -> inter-satellite spread is comparable/larger")

== maximum pairwise centroid spread across satellites (flux-weighted) ==
  top10_5deg_mean: 284 km
  top5_5deg_mean: 453 km
  top10_2deg_mean: 265 km
  top5_2deg_mean: 387 km
  NOAA-POES: centroids noaa15(-21.0,-52.9), noaa18(-20.9,-55.6), noaa19(-20.9,-55.5)
  MetOp: centroids metop01(-21.7,-55.0), metop03(-21.4,-55.6)

  NOAA-15 top10/5deg centroid distance to others: min 228 km, max 284 km

== magnitude check ==
  inter-satellite spread (top10 5deg mean): 284 km
  CP4B intra-satellite threshold shift top20->top1 (5deg mean): 433 km
  -> inter-satellite footprint spread is SMALL relative to the methodological threshold effect


## 7. Figures (no smoothing/interpolation; satellites + threshold/stat/grid labelled; blank = no data/masked)

In [7]:
# A/B per-satellite mean + sample-count maps (5deg)
for sat in compatible:
    plot_satellite_mean_map(grids[sat][5], 5.0, "enough_samples_5deg",
                            FIG / f"cp4f_{sat}_mean_flux_5deg.png", sat)
    plot_satellite_sample_count(grids[sat][5], 5.0, FIG / f"cp4f_{sat}_sample_count_5deg.png", sat)
# C/D overlays: top10 & top5 at 5deg and 2deg
for thr in ("top10", "top5"):
    for gd, mc in [(5.0, "enough_samples_5deg"), (2.0, "enough_samples_2deg")]:
        plot_satellite_comparison([(s, grids[s][int(gd)]) for s in compatible], gd, mc,
                                  FIG / f"cp4f_multisatellite_{thr}_{int(gd)}deg_mean_overlay.png", thr)
# E centroid comparison (top10+top5) ; F pairwise heatmap
plot_multisatellite_centroids(sens, FIG / "cp4f_multisatellite_centroid_comparison_top10_top5.png")
plot_pairwise_distance_heatmap(pairwise, "top10_5deg_mean",
                               FIG / "cp4f_pairwise_centroid_distance_top10_5deg_mean.png")
print("CP4F figures:", sorted(p.name for p in FIG.glob("cp4f_*.png")))

CP4F figures: ['cp4f_metop01_mean_flux_5deg.png', 'cp4f_metop01_sample_count_5deg.png', 'cp4f_metop03_mean_flux_5deg.png', 'cp4f_metop03_sample_count_5deg.png', 'cp4f_multisatellite_centroid_comparison_top10_top5.png', 'cp4f_multisatellite_top10_2deg_mean_overlay.png', 'cp4f_multisatellite_top10_5deg_mean_overlay.png', 'cp4f_multisatellite_top5_2deg_mean_overlay.png', 'cp4f_multisatellite_top5_5deg_mean_overlay.png', 'cp4f_noaa15_mean_flux_5deg.png', 'cp4f_noaa15_sample_count_5deg.png', 'cp4f_noaa18_mean_flux_5deg.png', 'cp4f_noaa18_sample_count_5deg.png', 'cp4f_noaa19_mean_flux_5deg.png', 'cp4f_noaa19_sample_count_5deg.png', 'cp4f_pairwise_centroid_distance_top10_5deg_mean.png']


## 8. Summary

- **Compatibility:** all five satellites (noaa15, noaa18, noaa19, metop01, metop03) re-confirmed —
  full Jan-2024 coverage, open with the loader, identical `mep_omni_flux_p1` units/long_name. None
  excluded.
- **Footprint consistency:** the candidate high-flux footprints **broadly overlap** across all
  satellites; the maximum pairwise flux-weighted centroid spread (top10/top5, 5deg/2deg) is printed in
  sections 5-6 and is **small relative to the CP4B intra-satellite threshold effect** — i.e. choosing a
  different satellite moves the footprint center far less than choosing a different flux threshold does.
- **NOAA vs MetOp families** look broadly consistent; **NOAA-15** (oldest) sits within the same cluster
  in footprint location (its absolute flux is not compared).
- **Absolute flux is NOT compared** (`absolute_flux_comparison_allowed=False`): differences may be
  calibration / instrument aging / detector response / orbital-sampling / platform, not physical.

This is a **calibration-limited, multi-satellite footprint-location consistency** result — *not* a true
SAA boundary/center, dose, health risk, danger zone, or discovery. `mep_IFC_on==-1` uninterpreted.